# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_attenuation
import snspd4
params = snspd4.snspd4()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260411-3440-qcodes.log
Experiment loaded. Last ID no: 462


# Instruments

In [11]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

In [13]:
MS = station.load_instrument("mso5", revive_instance=True)

# Light Counts

In [68]:
osc_set_standard(MS, v_scale=v_scale, h_time=params.h_time_counts, h_pos=params.h_pos_counts)

In [70]:
ID = params.att_info_id
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

In [71]:
v_attenuator

array([3.4 , 3.45, 3.5 , 3.55, 3.6 , 3.65, 3.7 , 3.75, 3.8 , 3.85, 3.9 ,
       3.95, 4.  , 4.05, 4.1 , 4.15, 4.2 , 4.25, 4.3 , 4.35, 4.4 , 4.45,
       4.5 , 4.55, 4.6 , 4.65, 4.7 , 4.75, 4.8 , 4.85, 4.9 , 4.95, 5.  ,
       5.05, 5.1 , 5.15, 5.2 , 5.25, 5.3 , 5.35, 5.4 , 5.45, 5.5 , 5.55,
       5.6 , 5.65, 5.7 , 5.75, 5.8 , 5.85, 5.9 , 5.95, 6.  , 6.05, 6.1 ,
       6.15, 6.2 , 6.25, 6.3 , 6.35, 6.4 , 6.45, 6.5 , 6.55, 6.6 , 6.65,
       6.7 , 6.75, 6.8 , 6.85, 6.9 , 6.95, 7.  ])

In [72]:
osc_set_standard(MS, v_scale=v_scale, h_time=params.h_time_counts, h_pos=params.h_pos_counts)

osc_check_standard(MS)

MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.15
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0


In [11]:
(len(v_attenuator)*10)/60

12.166666666666666

In [12]:
station.snapshot() # <- updates parameters in station

{'instruments': {},
 'parameters': {},
 'components': {},
 'config': {'instruments': {'mso5': {'type': 'MSO5.MSO5', 'address': 'TCPIP0::10.196.52.96::inst0::INSTR', 'init': {'channels_n': 4}}, 'dmm': {'type': 'qcodes.instrument_drivers.Keysight.Keysight34410A', 'address': 'TCPIP0::10.196.52.73::inst0::INSTR'}, 'fridge': {'type': 'fridge.FridgeTemps', 'init': {'url': 'https://qsyd.sydney.edu.au/data/Friesland?current'}}, 'yoko': {'type': 'qcodes.instrument_drivers.yokogawa.GS200.GS200', 'address': 'TCPIP0::10.196.52.75::inst0::INSTR', 'init': {'terminator': '\n'}}, 'pm100usb': {'type': 'Thorlabs_PM100.Thorlabs_PM100USB', 'address': 'USB0::0x1313::0x8072::1906768::0::INSTR', 'init': {'terminator': '\r\n'}}, 'pms120': {'type': 'Thorlabs_PM100.Thorlabs_S120', 'address': 'USB0::0x1313::0x8072::1913782::0::INSTR', 'init': {'terminator': '\r\n'}}, 'laser': {'type': 'PPCL550.PPCL550', 'address': 'ASRL13'}, 'dmm_keithley': {'type': 'Keithley_2000_new.Keithley2000', 'address': 'USB0::0x05E6::0x2

In [14]:
ID = params.att_info_id
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

# set to maximum voltage and maximum attenuation
p_att.write(f'VOLT {max(v_attenuator)}')
time.sleep(10)

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')

time.sleep(10)

if MS.channels[0].clipping(): 
    print('Error: Clipping')

snspd_counts_vs_attenuation(MS, dmm, yoko, p_att, device_name=params.device_1_name, n_captures=10, interval=1, current=12e-6, v_attenuator=v_attenuator, station=station)


############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
update station
Starting experimental run with id: 463. 
463
Current is 1.2e-05
Starting V=7
This acquisition will take 10s
15 58
Starting V=6.95
This acquisition will take 10s
15 58
Starting V=6.9
This acquisition will take 10s
15 58
Starting V=6.85
This acquisition will take 10s
15 58
Starting V=6.8
This acquisition will take 10s
15 59
Starting V=6.75
This acquisition will take 10s
15 59
Starting V=6.7
This acquisition will take 10s
15 59
Starting V=6.65
This acquisition will take 10s
15 59
Starting V=6.6
This acquisition will take 10s
16 0
Starting V=6.55
This acquisition will take 10s
16 0
Starting V=6.5
This acquisition will take 10s
16 0
Starting V=6.45
This acquisition will take 10s
16 1
Starting V=6.4
This acquisition will take 10s
16 1
Starting V=6.35
This acquisition will take 10s
16 1
Starting V=6.3
This acquisition will take 10s
16 1
Starting V=6.25
This acquisition will take 10s
16 2
Starting V=6.2
This acquisition will take 10s
16 2
Starting V=6.1

In [15]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


In [16]:
time.sleep(120)

In [17]:
ID = params.att_info_id
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

# set to maximum voltage and maximum attenuation
p_att.write(f'VOLT {max(v_attenuator)}')
time.sleep(10)

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')

time.sleep(10)

if MS.channels[0].clipping(): 
    print('Error: Clipping')

snspd_counts_vs_attenuation(MS, dmm, yoko, p_att, device_name=params.device_1_name, n_captures=10, interval=1, current=12e-6, v_attenuator=v_attenuator, station=station)


############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
update station
Starting experimental run with id: 464. 
464
Current is 1.2e-05
Starting V=7
This acquisition will take 10s
16 20
Starting V=6.95
This acquisition will take 10s
16 20
Starting V=6.9
This acquisition will take 10s
16 20
Starting V=6.85
This acquisition will take 10s
16 20
Starting V=6.8
This acquisition will take 10s
16 21
Starting V=6.75
This acquisition will take 10s
16 21
Starting V=6.7
This acquisition will take 10s
16 21
Starting V=6.65
This acquisition will take 10s
16 21
Starting V=6.6
This acquisition will take 10s
16 22
Starting V=6.55
This acquisition will take 10s
16 22
Starting V=6.5
This acquisition will take 10s
16 22
Starting V=6.45
This acquisition will take 10s
16 23
Starting V=6.4
This acquisition will take 10s
16 23
Starting V=6.35
This acquisition will take 10s
16 23
Starting V=6.3
This acquisition will take 10s
16 23
Starting V=6.25
This acquisition will take 10s
16 24
Starting V=6.2
This acquisition will take 10s
16 24
Start

In [18]:
laser.enable()

False